In [ ]:
# 10.03.24 Author: ArborRose
# edited by Jesse Tymas
# This Python code uses the Domo API to retrieve
# datasets based on the "tag". 

In [ ]:
import os
import json
import sys
import time
import requests
import pandas as pd
import domojupyter as domo
from dotenv import load_dotenv

# Configuration & Final Values

load_dotenv()
DOMO_INSTANCE = os.getenv("DOMO_INSTANCE")
DOMO_DEV_TOKEN = os.getenv("DOMO_DEV_TOKEN")
URL = f"https://{DOMO_INSTANCE}.domo.com/api/data/ui/v3/datasources/search"

DATASET_TAGS_DATASET_NAME = "Dataset Tag Relationships"
CARD_TAGS_DATASET_NAME = "Card Tag Relationships"

REQUEST_TIMEOUT_SEC = 20
FETCH_RETRIES = 3
FETCH_BACKOFF_SEC = 1

DEBUG = False

In [ ]:
# Finds a key in a dictionary
def find_key(d, key):
    if isinstance(d, dict):
        for k, v in d.items():
            if k == key:
                return v
            res = find_key(v, key)
            if res is not None:
                return res
    elif isinstance(d, list):
        for item in d:
            res = find_key(item, key)
            if res is not None:
                return res
    return None

In [ ]:
# Set headers for the API call
headers = {
    'X-DOMO-Developer-Token': DOMO_DEV_TOKEN,
    'Content-Type': 'application/json'
}

# Payload for searching datasets by tag
payload = {
    "entities": ["DATASET"],  # Specify that we want datasets
    "filters": [
        {
            "filterType": "facetValue",
            "field": "tag",
            "value": "*"  # Replace with your tag as needed
        }
    ],
    "combineResults": "true",
    "query": "*",  # Use wildcard to match all
    "count": 10000,  # Number of datasets to retrieve
    "offset": 0,  # Starting offset for pagination
    "sort": {
        "isRelevance": "false",
        "fieldSorts": [
            {
                "field": "create_date",
                "sortOrder": "DESC"  # Sort by creation date
            }
        ]
    }
}

# Make the API request to search for datasets
response = requests.post(URL, json=payload, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Print the full response to understand its structure
    response_data = response.json()
    if DEBUG:
        print(json.dumps(response_data, indent=2))  # Pretty-print the JSON response
else:
    print(f"Failed to fetch datasets: {response.status_code} - {response.text}")

# Save response data to df
df = pd.DataFrame(response_data['dataSources'])

In [ ]:
# Create a row per tag relationship
tag_relationships = (
    df.explode("tagsList", ignore_index=True)
      .rename(columns={"tagsList": "tag"})
)

In [ ]:
domo.write_dataframe(tag_relationships, "Monitoring | Dataset Tags")